In [ ]:
from pathlib import Path

from icalens import ICALens

HERE = Path.cwd()
if not (HERE / "output" / "ica-lens-profiled").exists():
    HERE = HERE / "experiments" / "liusida" / "superposition"
lens = ICALens.from_pretrained(HERE / "output" / "ica-lens-profiled")
prompt = "The artist painted a red square."

In [ ]:
analysis = lens.analyze(
    prompt,
    layer=5,
)

analysis

In [ ]:
from pathlib import Path

import torch

for layer in lens.available_layers:
    result = lens.analyze(prompt, layer=layer, device="cuda")
    target = next(
        i for i, text in enumerate(result.token_texts)
        if text.strip() == "square"
    )

    scores = result.scores[target]
    components = torch.topk(scores.abs(), k=10).indices.tolist()

    print(f"\n{'=' * 20} Layer {layer} {'=' * 20}")

    for component in components:
        score = scores[component].item()
        sign = "positive" if score >= 0 else "negative"
        profile = lens.component_profile(layer=layer, component=component)
        occurrences = profile["examples"][sign]["occurrences"][:5]

        print(f"\nC{component}: score={score:+.3f}, rank sign={sign}")

        for occurrence in occurrences:
            print(
                f"  {occurrence['text']!r} "
                f"(profile score={occurrence['score']:+.3f}) — "
                f"{occurrence['context']!r}"
            )

